In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier

from tqdm import tqdm
import os
os.chdir('E:/Machine_Learning/Cases/human-resources-analytics/')

In [3]:
df = pd.read_csv('HR_comma_sep.csv')
df.head()


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low


In [5]:
X , y = df.drop('left',axis =1), df['left']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

ohe = OneHotEncoder(sparse_output= False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')
X_train_ohe = ct.fit_transform(X_train)
X_test_ohe = ct.transform(X_test)

In [8]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = GradientBoostingClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train_ohe,y_train)
            y_pred = gbm.predict(X_test_ohe)
            scores.append([l,m,n,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','accuracy'])
df_scores.sort_values(by='accuracy',ascending=False)

100%|██████████| 10/10 [02:03<00:00, 12.36s/it]


,Learning_rate,Max_depth,n_estimators,accuracy
83,0.603333,4.0,100,0.987553
71,0.504444,4.0,100,0.985552
82,0.603333,4.0,50,0.983774
1,0.010000,NaN,50,0.983552
59,0.405556,4.0,100,0.983330
...,...,...,...,...
4,0.010000,2.0,50,0.762392
6,0.010000,3.0,25,0.762169
3,0.010000,2.0,25,0.762169
0,0.010000,NaN,25,0.762169


### The execution is taking too much time .. trying XGBoost

In [11]:
from xgboost import XGBClassifier

In [12]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = XGBClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train_ohe,y_train)
            y_pred = gbm.predict(X_test_ohe)
            scores.append([l,m,n,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','accuracy'])
df_scores.sort_values(by='accuracy',ascending=False)

100%|██████████| 10/10 [00:18<00:00,  1.89s/it]


,Learning_rate,Max_depth,n_estimators,accuracy
62,0.504444,NaN,100,0.988220
38,0.306667,NaN,100,0.987997
73,0.603333,NaN,50,0.987775
85,0.702222,NaN,50,0.987775
86,0.702222,NaN,100,0.987553
...,...,...,...,...
4,0.010000,2.0,50,0.762392
6,0.010000,3.0,25,0.762169
3,0.010000,2.0,25,0.762169
0,0.010000,NaN,25,0.762169


## XGBoost only takes 18 seconds to run the code
In short terms, XGBoost is like an upgraded, turbocharged version of standard Gradient Boosting. Here are the four biggest differences:

Built-in Regularization (L1 & L2): Standard gradient boosting relies mostly on tree depth to prevent overfitting. XGBoost adds mathematical penalties directly to the model to punish complexity, making it much harder to overfit.

Automatic Missing Value Handling: Standard boosting often crashes or requires manual filling (imputation) of missing data. XGBoost automatically learns the best "default direction" to send missing values during splits.

Faster, Parallel Processing: Standard gradient boosting builds splits sequentially. XGBoost parallelizes the work across your CPU cores, making it significantly faster.

Smart Large-Data Handling: When datasets get massive, sorting data to find splits slows traditional algorithms down. XGBoost uses smart approximations to process huge datasets in a fraction of the time.

------
# LightGBM

In [13]:
from lightgbm import LGBMClassifier

In [14]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = LGBMClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train_ohe,y_train)
            y_pred = gbm.predict(X_test_ohe)
            scores.append([l,m,n,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','accuracy'])
df_scores.sort_values(by='accuracy',ascending=False)

  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 2497, number of negative: 7999
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000883 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 10496, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.237900 -> initscore=-1.164227
[LightGBM] [Info] Start training from score -1.164227
[LightGBM] [Info] Number of positive: 2497, number of negative: 7999
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000362 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 10496, number of used features: 18
[LightGBM] [Info] [binar

 10%|█         | 1/10 [00:00<00:05,  1.68it/s]

[LightGBM] [Info] Number of positive: 2497, number of negative: 7999
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000465 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 10496, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.237900 -> initscore=-1.164227
[LightGBM] [Info] Start training from score -1.164227
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

 20%|██        | 2/10 [00:01<00:04,  1.69it/s]

...

 90%|█████████ | 9/10 [00:05<00:00,  1.71it/s]

[LightGBM] [Info] Number of positive: 2497, number of negative: 7999
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 10496, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.237900 -> initscore=-1.164227
[LightGBM] [Info] Start training from score -1.164227
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

100%|██████████| 10/10 [00:05<00:00,  1.70it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,Learning_rate,Max_depth,n_estimators,accuracy
37,0.306667,NaN,50,0.988664
26,0.207778,NaN,100,0.988664
50,0.405556,NaN,100,0.988442
38,0.306667,NaN,100,0.988442
61,0.504444,NaN,50,0.988442
...,...,...,...,...
4,0.010000,2.0,50,0.762392
6,0.010000,3.0,25,0.762169
3,0.010000,2.0,25,0.762169
0,0.010000,NaN,25,0.762169


# CatBoost 

In [15]:
from catboost import CatBoostClassifier

In [17]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostClassifier(random_state=26,verbose=False,learning_rate=l,max_depth=m,n_estimators=n)
            cat.fit(X_train_ohe,y_train)
            y_pred = cat.predict(X_test_ohe)
            scores.append([l,m,n,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','accuracy'])
df_scores.sort_values(by='accuracy',ascending=False)

100%|██████████| 10/10 [00:34<00:00,  3.49s/it]


,Learning_rate,Max_depth,n_estimators,accuracy
98,0.801111,NaN,100,0.981551
110,0.900000,NaN,100,0.980440
86,0.702222,NaN,100,0.979773
119,0.900000,4.0,100,0.979106
74,0.603333,NaN,100,0.978884
...,...,...,...,...
5,0.010000,2.0,100,0.907090
7,0.010000,3.0,50,0.906646
6,0.010000,3.0,25,0.901534
4,0.010000,2.0,50,0.856635


In [19]:
X.dtypes

satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
promotion_last_5years      int64
Department                   str
salary                       str
dtype: object

#### In catboosting we don't need to encode the categotical data 
#### It automatically handles categorical variables out-of-the-box. It uses a clever mathematical trick called Ordered Target Encoding

In [20]:
catagorical = ['salary','Department']
cat = CatBoostClassifier(random_state=26,verbose=False,learning_rate=l,max_depth=m,n_estimators=n, =catagorical)
cat.fit(X_train,y_train)
y_pred = cat.predict(X_test)
accuracy_score(y_test,y_pred)

0.975994665481218

### using this in loop

In [21]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostClassifier(random_state=26,verbose=False,learning_rate=l,max_depth=m,n_estimators=n)
            cat2 = CatBoostClassifier(random_state=26,verbose=False,learning_rate=l,max_depth=m,n_estimators=n,cat_features=catagorical)
            cat.fit(X_train_ohe,y_train)
            y_pred = cat.predict(X_test_ohe)
            cat2.fit(X_train,y_train)
            y_pred2 = cat2.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test,y_pred),accuracy_score(y_test,y_pred2)])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','accuracy-with-OHE','accuracy-without-OHE'])
df_scores.nlargest(5,'accuracy-with-OHE')


100%|██████████| 10/10 [02:43<00:00, 16.35s/it]


,Learning_rate,Max_depth,n_estimators,accuracy-with-OHE,accuracy-without-OHE
110,0.900000,NaN,100,0.980440,0.980885
98,0.801111,NaN,100,0.981551,0.979773
86,0.702222,NaN,100,0.979773,0.978662
97,0.801111,NaN,50,0.977328,0.978440
107,0.801111,4.0,100,0.978440,0.978217


In [22]:
df_scores.nlargest(5,'accuracy-without-OHE')

,Learning_rate,Max_depth,n_estimators,accuracy-with-OHE,accuracy-without-OHE
110,0.900000,NaN,100,0.980440,0.980885
98,0.801111,NaN,100,0.981551,0.979773
86,0.702222,NaN,100,0.979773,0.978662
97,0.801111,NaN,50,0.977328,0.978440
107,0.801111,4.0,100,0.978440,0.978217
